# Neuron bridge, step 3: global per-cell W shift

Measure. For every cell we compute a single 3'UTR-length index W in [0, 1]: for
each multi-site gene the cell expresses, W_gene = sum_k r_k * p_k where r_k is the within-gene
rank-normalised position of poly(A) site k (0 = shortest 3'UTR, 1 = longest) and p_k its usage
fraction in that cell; the cell's global W is the mean of W_gene over all genes it expresses
(median ~125 genes/cell here). Higher global W = the cell uses longer 3'UTRs on average.

Main readout. Global W, rather than per-gene Delta-psi, is the main readout of the neuron
comparison (Fig. 1G). It averages over ~125 genes per cell, so it is not affected by the per-gene
sparsity that limits the neuron group (step 1). It measures whether neurons use longer or shorter
3'UTRs than neuroblasts overall, the meta-gene analogue of the global 3'UTR-lengthening readout
used for the ASD cortex data.

Statistics. W is a bounded effect-size descriptor. We report the distribution shift and a
non-parametric effect size (Cliff's delta, equivalently rank-biserial = 2*AUC - 1), and we check
per-batch reproducibility (batches 3 and 4 independently). We do not report a per-cell p-value
(cells are pseudoreplicates).

Output: `DirMult_neuron_bridge/neuron_W_global_shift.csv` (group summary) and
`neuron_W_global_percell.csv` (per-cell values), plus figures.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026/05-neuron_bridge")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from scipy.stats import rankdata
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.dpi": 100,
        "savefig.dpi": 150,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

BASE = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"
APA = f"{BASE}/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
OUT = f"{BASE}/DirMult_neuron_bridge"
FIG = f"{OUT}/figures"
os.makedirs(FIG, exist_ok=True)
LINE = ["qNSC1", "qNSC2", "aNSC1", "aNSC2", "TAP", "NB", "Neurons"]
COL = dict(zip(LINE, ["#4575b4", "#74add1", "#abd9e9", "#fee090", "#fdae61", "#f46d43", "#d73027"]))
BATCHES = ["3", "4"]


def cliff_delta(x_late, x_early):
    """Cliff's delta = rank-biserial = 2*AUC-1; + => x_late stochastically larger (longer 3'UTRs)."""
    n1, n2 = len(x_late), len(x_early)
    rk = rankdata(np.concatenate([x_late, x_early]))
    U1 = rk[:n1].sum() - n1 * (n1 + 1) / 2.0
    return 2.0 * (U1 / (n1 * n2)) - 1.0


# ---- compute per-cell global W for the lineage + neurons (batches 3+4) ----
a = ad.read_h5ad(APA)
ct = a.obs["celltype"].astype(str).values
bt = a.obs["batch_number"].astype(str).values
inb = np.isin(bt, BATCHES)
X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)

v = a.var
rs = v.groupby("utr_name")["utr_size"].rank(method="average")  # within-gene rank by UTR length
K = v.groupby("utr_name")["utr_size"].transform("size")  # sites per gene
r = np.nan_to_num(
    ((rs - 1) / np.where(K.values > 1, K.values - 1, 1)).values, nan=0.0
)  # r_k in [0,1]
genes = v["utr_name"].astype(str).values
uniq, inv = np.unique(genes, return_inverse=True)
multi = np.bincount(inv) >= 2
G = sp.csr_matrix((np.ones(len(inv)), (np.arange(len(inv)), inv)), shape=(len(inv), len(uniq)))
Gm = G[:, multi]  # features -> multi-site genes

mask = np.isin(ct, LINE) & inb
Xs = X[mask]
num = np.asarray(
    (Xs.multiply(r.reshape(1, -1)) @ Gm).todense()
)  # sum_k r_k count_k per (cell,gene)
den = np.asarray((Xs @ Gm).todense())  # total count per (cell,gene)
with np.errstate(invalid="ignore", divide="ignore"):
    ratio = np.where(den > 0, num / den, np.nan)  # W_gene per (cell,gene)
W = np.nanmean(ratio, axis=1)  # global W per cell
percell = pd.DataFrame(
    {
        "cell_id": a.obs_names[mask],
        "celltype": ct[mask],
        "batch": bt[mask],
        "global_W": W,
        "n_genes": (den > 0).sum(1),
    }
)
percell.to_csv(f"{OUT}/neuron_W_global_percell.csv", index=False)
print(
    f"per-cell global W computed for {len(percell)} cells (median {int(percell.n_genes.median())} genes/cell)"
)

## Section A: global W along the lineage and the neuron jump

The lineage (qNSC -> NB) shows a mild downward drift in global W (a weak shortening trend,
consistent with the small proliferation-linked shortening bias reported for Fig. 1). The terminal
Neurons break that trend with a clear upward step: they use longer 3'UTRs than any progenitor
stage. The violins show a whole-distribution shift, not an outlier effect.

In [ ]:
g = percell.groupby("celltype")["global_W"]
summ = g.agg(["count", "mean", "median", "std"]).reindex(LINE)
summ["sem"] = summ["std"] / np.sqrt(summ["count"])
display(summ.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
# (left) mean +/- SEM along lineage, neurons as the terminal discrete point
xpos = np.arange(len(LINE))
ax[0].errorbar(
    xpos[:-1],
    summ["mean"].values[:-1],
    yerr=summ["sem"].values[:-1],
    fmt="o-",
    color="#555",
    capsize=3,
    label="lineage (qNSC->NB)",
)
ax[0].errorbar(
    xpos[-1],
    summ["mean"].values[-1],
    yerr=summ["sem"].values[-1],
    fmt="D",
    ms=10,
    color=COL["Neurons"],
    capsize=3,
    label="Neurons (discrete)",
)
ax[0].axvspan(xpos[-1] - 0.5, xpos[-1] + 0.5, color=COL["Neurons"], alpha=0.07)
ax[0].set_xticks(xpos)
ax[0].set_xticklabels(LINE, rotation=45)
ax[0].set(
    ylabel="mean global W (3'UTR-length index)", title="Global 3'UTR length across the lineage"
)
ax[0].legend(fontsize=8, frameon=False)
# (right) per-cell W distributions
data = [percell.loc[percell.celltype == l, "global_W"].values for l in LINE]
parts = ax[1].violinplot(data, showmedians=True, widths=0.85)
for i, b in enumerate(parts["bodies"]):
    b.set_facecolor(COL[LINE[i]])
    b.set_alpha(0.8)
ax[1].set_xticks(np.arange(1, len(LINE) + 1))
ax[1].set_xticklabels(LINE, rotation=45)
ax[1].set(ylabel="per-cell global W", title="Per-cell W distribution by stage")
fig.tight_layout()
fig.savefig(f"{FIG}/s3A_global_W_lineage.png", bbox_inches="tight")
plt.show()

## Section B: effect size and per-batch reproducibility

We quantify the NB -> neuron (and qNSC -> neuron) shift with Cliff's delta (a non-parametric
effect size in [-1, 1]; + means neurons use longer 3'UTRs). We then split by batch: the neuron group
lives only in batches 3 and 4, so a shift in the same direction in each batch guards against a
single-batch artefact (an n = 2 test is not meaningful, so we report direction and effect size).

In [ ]:
wn = percell.loc[percell.celltype == "Neurons", "global_W"].values
wb = percell.loc[percell.celltype == "NB", "global_W"].values
wq = percell.loc[percell.celltype.isin(["qNSC1", "qNSC2"]), "global_W"].values
d_nb = cliff_delta(wn, wb)
d_q = cliff_delta(wn, wq)
print(f"NB   -> Neuron : meanW {wb.mean():.4f} -> {wn.mean():.4f}   Cliff's delta = {d_nb:+.3f}")
print(f"qNSC -> Neuron : meanW {wq.mean():.4f} -> {wn.mean():.4f}   Cliff's delta = {d_q:+.3f}")

rows = [
    {
        "contrast": "NB->Neuron",
        "cliff_delta": d_nb,
        "meanW_early": wb.mean(),
        "meanW_late": wn.mean(),
        "n_early": len(wb),
        "n_late": len(wn),
    },
    {
        "contrast": "qNSC->Neuron",
        "cliff_delta": d_q,
        "meanW_early": wq.mean(),
        "meanW_late": wn.mean(),
        "n_early": len(wq),
        "n_late": len(wn),
    },
]
perbatch = []
for b in BATCHES:
    A = percell[(percell.celltype == "Neurons") & (percell.batch == b)].global_W
    B = percell[(percell.celltype == "NB") & (percell.batch == b)].global_W
    perbatch.append(
        {
            "batch": b,
            "NB_meanW": B.mean(),
            "neuron_meanW": A.mean(),
            "diff_late_minus_early": A.mean() - B.mean(),
            "n_NB": len(B),
            "n_neuron": len(A),
        }
    )
perbatch = pd.DataFrame(perbatch)
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/neuron_W_global_shift.csv", index=False)
perbatch.to_csv(f"{OUT}/neuron_W_global_shift_perbatch.csv", index=False)
print("\nper-batch NB vs neuron (reproducibility):")
display(perbatch.round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(
    [wb, wn],
    labels=["NB", "Neurons"],
    widths=0.6,
    showfliers=False,
    patch_artist=True,
    boxprops=dict(facecolor="#eee"),
    medianprops=dict(color="k"),
)
for i, (arr, c) in enumerate([(wb, COL["NB"]), (wn, COL["Neurons"])], start=1):
    jit = np.random.RandomState(0).normal(i, 0.05, size=len(arr))
    ax[0].scatter(jit, arr, s=4, c=c, alpha=0.25, linewidths=0)
ax[0].set(ylabel="per-cell global W", title=f"NB vs Neuron  (Cliff's d = {d_nb:+.2f})")
x = np.arange(len(perbatch))
ax[1].bar(x - 0.18, perbatch["NB_meanW"], 0.36, label="NB", color=COL["NB"])
ax[1].bar(x + 0.18, perbatch["neuron_meanW"], 0.36, label="Neurons", color=COL["Neurons"])
ax[1].set_xticks(x)
ax[1].set_xticklabels("batch " + perbatch["batch"])
ax[1].set(ylabel="mean global W", title="Per-batch means (same direction in both)")
ax[1].legend(fontsize=8, frameon=False)
ax[1].set_ylim(0.44, 0.54)
fig.tight_layout()
fig.savefig(f"{FIG}/s3B_W_effectsize_perbatch.png", bbox_inches="tight")
plt.show()

## Summary

- Neurons use longer 3'UTRs than neuroblasts: global W rises from ~0.468 (NB) to ~0.518
  (Neurons), Cliff's delta = +0.56 (a large effect), and the shift is positive in batch 3 and
  batch 4 independently.
- This reverses the mild shortening drift of the qNSC -> NB lineage and matches the known
  neuronal 3'UTR-lengthening programme (Miura et al. 2013), with lengthening at terminal neuronal
  differentiation.
- As an all-gene, per-cell meta-gene measure, it is not affected by the per-gene coverage limit.
  The next steps ask which genes drive it
  (Step 4: continuation vs reversal vs the main trajectory) and whether the same direction holds in
  the human organoid neuron trajectory (Step 5: cross-species).